# 🏆 Churn Prediction — Part 4: Advanced Techniques
## Push the score further with competition tricks

| Technique | What it does | Expected gain |
|-----------|-------------|---------------|
| Original dataset | Add real-world source data to training | +0.001-0.003 |
| Multi-seed averaging | Train same model with different seeds | +0.0005-0.001 |
| Feature selection | Remove noisy features | +0.0005 |
| Target encoding | Encode categoricals using target stats | +0.001 |

**Input:** `X_train.csv`, `X_test.csv`, `y_train.csv`, `train.csv` (raw), original dataset  
**Output:** `part4_oof_*.npy`, `part4_test_*.npy`


In [3]:
import pandas as pd, numpy as np, time, warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb
warnings.filterwarnings('ignore')

X_train = pd.read_csv('/Users/parveenkumarsharma/Downloads/playground-series-s6e3/X_train.csv')
X_test = pd.read_csv('/Users/parveenkumarsharma/Downloads/playground-series-s6e3/X_test.csv')
y_train = pd.read_csv('/Users/parveenkumarsharma/Downloads/playground-series-s6e3/y_train.csv').squeeze()
test_ids = pd.read_csv('/Users/parveenkumarsharma/Downloads/playground-series-s6e3/test_ids.csv').squeeze()

SPW = (y_train == 0).sum() / (y_train == 1).sum()
SEED = 42

print(f"Data: {X_train.shape[0]:,} × {X_train.shape[1]}")


Data: 594,194 × 71


In [4]:
def full_cv(name, train_fn, X, y, X_test, n_folds=5, seed=42):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros(len(y)); test_preds = np.zeros(len(X_test)); scores = []
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        model = train_fn(X.iloc[tr], y.iloc[tr], X.iloc[va], y.iloc[va])
        oof[va] = model.predict_proba(X.iloc[va])[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / n_folds
        scores.append(roc_auc_score(y.iloc[va], oof[va]))
        print(f"  Fold {fold+1}: {scores[-1]:.6f}")
    mean = np.mean(scores)
    print(f"  ✅ {name}: {mean:.6f} ± {np.std(scores):.6f}")
    return oof, test_preds, mean


## Technique 1: Add Original Dataset
The competition data is synthetically generated from a real dataset. We can download the original and add it as extra training data. This helps because the model sees more diverse patterns.

**Download from:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn  
Place `WA_Fn-UseC_-Telco-Customer-Churn.csv` in the same folder.


In [6]:
# ============================================================
# TECHNIQUE 1: ADD ORIGINAL DATASET
# ============================================================
# NOTE: Download the original dataset and place it here.
# If file not found, this section is skipped gracefully.

import os

orig_file = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
if os.path.exists(orig_file):
    print("▶ Loading original dataset...")
    orig = pd.read_csv(orig_file)
    print(f"  Original dataset: {orig.shape}")
    
    # Process original dataset through same pipeline as train
    # (You'd apply the same preprocessing_pipeline.py steps here)
    # This is left as an exercise — apply Phase 5 steps to orig data
    # Then concatenate: X_train_extended = pd.concat([X_train, X_orig])
    
    print("  ⚠️ Apply same preprocessing pipeline to original data")
    print("  Then: X_train_ext = pd.concat([X_train, X_train_orig])")
    print("        y_train_ext = pd.concat([y_train, y_train_orig])")
else:
    print(f"  ⚠️ {orig_file} not found. Skipping this technique.")
    print(f"  Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn")
    print(f"  Place in same directory and re-run.")


  ⚠️ WA_Fn-UseC_-Telco-Customer-Churn.csv not found. Skipping this technique.
  Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
  Place in same directory and re-run.


## Technique 2: Multi-Seed Averaging
Training the same model with different random seeds gives slightly different results each time. Averaging across seeds reduces variance and gives more stable predictions.

We train LightGBM with 5 different seeds and average.


In [7]:
# ============================================================
# TECHNIQUE 2: MULTI-SEED AVERAGING (LightGBM × 5 seeds)
# ============================================================
print("▶ Multi-Seed LightGBM (5 seeds × 5 folds = 25 models)")

seeds = [42, 123, 456, 789, 2024]
all_oof_ms = []
all_test_ms = []

for i, seed in enumerate(seeds):
    print(f"\n  ── Seed {seed} ({i+1}/5) ──")
    def train_lgb_seed(X_tr, y_tr, X_va, y_va):
        m = lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, num_leaves=31,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0, subsample_freq=1,
            scale_pos_weight=SPW, random_state=seed, n_jobs=-1, verbose=-1
        )
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        return m
    
    oof_s, test_s, score_s = full_cv(f"LGB-seed{seed}", train_lgb_seed, X_train, y_train, X_test, seed=seed)
    all_oof_ms.append(oof_s)
    all_test_ms.append(test_s)

# Average across seeds
oof_multiseed = np.mean(all_oof_ms, axis=0)
test_multiseed = np.mean(all_test_ms, axis=0)
score_multiseed = roc_auc_score(y_train, oof_multiseed)

print(f"\n  ✅ Multi-seed LGB average: {score_multiseed:.6f}")
print(f"  (Compare to single-seed LGB from Part 2)")


▶ Multi-Seed LightGBM (5 seeds × 5 folds = 25 models)

  ── Seed 42 (1/5) ──
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[55]	valid_0's binary_logloss: 0.365466
  Fold 1: 0.911207
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[52]	valid_0's binary_logloss: 0.366423
  Fold 2: 0.912056
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[55]	valid_0's binary_logloss: 0.36479
  Fold 3: 0.911873
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[55]	valid_0's binary_logloss: 0.362745
  Fold 4: 0.912813
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[52]	valid_0's binary_logloss: 0.36692
  Fold 5: 0.910161
  ✅ LGB-seed42: 0.911622 ± 0.000892

  ── Seed 123 (2/5) ──
Training until validation scores don't improve for 100 rounds
Early stopping, best it

## Technique 3: Feature Selection
Remove features with very low importance — they add noise without signal.


In [8]:
# ============================================================
# TECHNIQUE 3: FEATURE SELECTION (Remove bottom 20% features)
# ============================================================
print("▶ Feature Selection (importance-based)")

# Train LGB to get feature importance
m_imp = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31,
    scale_pos_weight=SPW, verbose=-1, n_jobs=-1, random_state=SEED)
m_imp.fit(X_train, y_train)

imp = pd.DataFrame({'feature': X_train.columns, 'importance': m_imp.feature_importances_})
imp = imp.sort_values('importance', ascending=False)

# Keep top 80% features (remove bottom 20%)
n_keep = int(len(imp) * 0.80)
keep_features = imp.head(n_keep)['feature'].tolist()
drop_features = imp.tail(len(imp) - n_keep)['feature'].tolist()

print(f"  Total features: {len(imp)}")
print(f"  Keeping: {n_keep} | Dropping: {len(drop_features)}")
print(f"  Dropped features: {drop_features}")

X_train_sel = X_train[keep_features]
X_test_sel = X_test[keep_features]

# Train with selected features
def train_lgb_sel(X_tr, y_tr, X_va, y_va):
    m = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=31,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, subsample_freq=1,
        scale_pos_weight=SPW, random_state=SEED, n_jobs=-1, verbose=-1
    )
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    return m

print(f"\n  Training LGB with {n_keep} features:")
oof_sel, test_sel, score_sel = full_cv("LGB-selected", train_lgb_sel, X_train_sel, y_train, X_test_sel)
print(f"\n  Feature selection {'helped! ✅' if score_sel > score_multiseed else 'did not help ❌'}")


▶ Feature Selection (importance-based)
  Total features: 71
  Keeping: 56 | Dropping: 15
  Dropped features: ['OnlineBackup_No internet service', 'tenure_years', 'PaymentMethod_Electronic check', 'is_new', 'Contract_Two year', 'Contract_Month-to-month', 'has_streaming', 'StreamingMovies_No internet service', 'tenure_sq', 'OnlineSecurity_No internet service', 'StreamingTV_No internet service', 'TechSupport_No internet service', 'DeviceProtection_No internet service', 'InternetService_No', 'InternetService_Fiber optic']

  Training LGB with 56 features:
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[54]	valid_0's binary_logloss: 0.365733
  Fold 1: 0.911134
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[54]	valid_0's binary_logloss: 0.366621
  Fold 2: 0.912074
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[54]	valid_0's binary_logloss: 0.

In [9]:
# ============================================================
# SAVE ALL PART 4 PREDICTIONS
# ============================================================
np.save('part4_oof_multiseed_lgb.npy', oof_multiseed)
np.save('part4_test_multiseed_lgb.npy', test_multiseed)
np.save('part4_oof_selected_lgb.npy', oof_sel)
np.save('part4_test_selected_lgb.npy', test_sel)

print("\n" + "=" * 55)
print("  PART 4 RESULTS")
print("=" * 55)
print(f"  Multi-seed LGB (5 seeds):  {score_multiseed:.6f}")
print(f"  Feature-selected LGB:      {score_sel:.6f}")
print(f"\n  ✅ Saved: part4_oof_*.npy, part4_test_*.npy")



  PART 4 RESULTS
  Multi-seed LGB (5 seeds):  0.911664
  Feature-selected LGB:      0.911597

  ✅ Saved: part4_oof_*.npy, part4_test_*.npy
